# DNABERT Dataset Preparation

This notebook is demonstrates how to take the ensembl enhancer genomic_benchmarks dataset from Hugging Face and prepare specific datasets for use as input to DNABERT for the genome-head-interpreter project.

### Acquire data

In [1]:
!pip install -qq datasets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.0/542.0 kB 5.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 5.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 8.1 MB/s eta 0:00:00


Access the data using HuggingFace:



In [2]:
from huggingface_hub import notebook_login

notebook_login()

In [3]:
import pandas as pd

In [4]:
from datasets import Dataset
from datasets import load_dataset
from sklearn.utils import shuffle


In [5]:
dataset_human_enhancer = load_dataset('katarinagresova/Genomic_Benchmarks_human_enhancers_ensembl', use_auth_token=True)

/usr/local/lib/python3.10/dist-packages/datasets/load.py:2547: FutureWarning: 'use_auth_token' was deprecated in favor of 'token' in version 2.14.0 and will be removed in 3.0.0.
You can remove this warning by passing 'token=<use_auth_token>' instead.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating train split:   0%|          | 0/123872 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/30970 [00:00<?, ? examples/s]

In [6]:
dataset_human_enhancer

DatasetDict({
    train: Dataset({
        features: ['seq', 'label'],
        num_rows: 123872
    })
    test: Dataset({
        features: ['seq', 'label'],
        num_rows: 30970
    })
})

### Inspect data to see what must be done to prepare for DNABERT

In [7]:
def count_sequences_with_n(data):
    count = 0
    for seq in data['seq']:
        if 'N' in seq:
            count += 1
    return count

# count sequences with 'N' in the training dataset
train_count = count_sequences_with_n(dataset_human_enhancer['train'])
print(f"Number of sequences with 'N' in train dataset: {train_count}")

# count sequences with 'N' in the testing dataset
test_count = count_sequences_with_n(dataset_human_enhancer['test'])
print(f"Number of sequences with 'N' in test dataset: {test_count}")


Number of sequences with 'N' in train dataset: 66
Number of sequences with 'N' in test dataset: 10


In [8]:
# function to compute sequence length
def compute_seq_length(example):
    example['length'] = len(example['seq'])
    return example


Minimum length sequence in the dataset?

In [9]:
# apply the function to the train and test datasets for enhancer
train_dataset_enhancer = dataset_human_enhancer['train'].map(compute_seq_length)
test_dataset_enhancer = dataset_human_enhancer['test'].map(compute_seq_length)

# average sequence length
avg_train_length_enhancer = sum(train_dataset_enhancer['length']) / len(train_dataset_enhancer)
avg_test_length_enhancer = sum(test_dataset_enhancer['length']) / len(test_dataset_enhancer)

print(f"Average train sequence length: {avg_train_length_enhancer}")
print(f"Average test sequence length: {avg_test_length_enhancer}")

Map:   0%|          | 0/123872 [00:00<?, ? examples/s]

Map:   0%|          | 0/30970 [00:00<?, ? examples/s]

Average train sequence length: 269.1078532678894
Average test sequence length: 267.88931223764934


In [10]:
print(min(train_dataset_enhancer['length']))

2


### Prepare data for DNABERT

Prepare functions and imports we will need for filtering

In [11]:
def filter_seq_length(example):
    return (len(example['seq']) >= 50) and ('N' not in example['seq']) and (len(example['seq']) <= 510)

In [12]:
#Original DNABERT function:

def seq2kmer(seq, k):
    """
    Convert original sequence to kmers

    Arguments:
    seq -- str, original sequence.
    k -- int, kmer of length k specified.

    Returns:
    kmers -- str, kmers separated by space

    """
    kmer = [seq[x:x+k] for x in range(len(seq)+1-k)]
    kmers = " ".join(kmer)
    return kmers

In [13]:
import numpy as np

### Prepare enhancer

Filter the data to make sure there are no sequences less than 205 in length and also remove any sequences with "N" characters

In [14]:
# filter the datasets
filtered_train_dataset_enhancer = dataset_human_enhancer['train'].filter(filter_seq_length)
filtered_test_dataset_enhancer = dataset_human_enhancer['test'].filter(filter_seq_length)

Filter:   0%|          | 0/123872 [00:00<?, ? examples/s]

Filter:   0%|          | 0/30970 [00:00<?, ? examples/s]

In [15]:
# apply the function to the filtered datasets
filtered_train_dataset_enhancer = filtered_train_dataset_enhancer.map(compute_seq_length)
filtered_test_dataset_enhancer = filtered_test_dataset_enhancer.map(compute_seq_length)

# average sequence length
avg_filtered_train_length_enhancer = sum(filtered_train_dataset_enhancer['length']) / len(filtered_train_dataset_enhancer)
avg_filtered_test_length_enhancer = sum(filtered_test_dataset_enhancer['length']) / len(filtered_test_dataset_enhancer)

print(f"Average filtered train sequence length: {avg_filtered_train_length_enhancer}")
print(f"Average filtered test sequence length: {avg_filtered_test_length_enhancer}")

Map:   0%|          | 0/115595 [00:00<?, ? examples/s]

Map:   0%|          | 0/28941 [00:00<?, ? examples/s]

Average filtered train sequence length: 271.1897573424456
Average filtered test sequence length: 270.7583013717563


In [16]:
# count sequences with 'N' in the training dataset after filtering...
train_count = count_sequences_with_n(filtered_train_dataset_enhancer)
print(f"Number of sequences with 'N' in train dataset: {train_count}")

# count sequences with 'N' in the testing dataset after filtering...
test_count = count_sequences_with_n(filtered_test_dataset_enhancer)
print(f"Number of sequences with 'N' in test dataset: {test_count}")


Number of sequences with 'N' in train dataset: 0
Number of sequences with 'N' in test dataset: 0


Confirm filtering:

In [18]:
print(min(filtered_train_dataset_enhancer['length']))
print(min(filtered_test_dataset_enhancer['length']))

50
50


In [20]:
print(max(filtered_train_dataset_enhancer['length']))
print(max(filtered_test_dataset_enhancer['length']))

510
510


Make the datasets by k-merizing using DNABERT k-merizing function, given the new filtering for sequences without "N" nucleotides and with sequence length >= 50 and =< 510:

In [21]:
train_human_enhancer = pd.DataFrame({"label": filtered_train_dataset_enhancer["label"]})
train_human_enhancer['sequence']= [seq2kmer(item, 6) for item in filtered_train_dataset_enhancer["seq"]]
test_human_enhancer = pd.DataFrame({"label": filtered_test_dataset_enhancer["label"]})
test_human_enhancer['sequence'] = [seq2kmer(item, 6) for item in filtered_test_dataset_enhancer["seq"]]

In [22]:
test_human_enhancer = shuffle(test_human_enhancer)
train_human_enhancer = shuffle(train_human_enhancer)

In [23]:
train_human_enhancer

,label,sequence
83233,1,GTGCAG TGCAGA GCAGAA CAGAAA AGAAAG GAAAGA AAAG...
54562,0,TGGGGG GGGGGC GGGGCA GGGCAG GGCAGG GCAGGG CAGG...
56087,0,GCAAAC CAAACA AAACAT AACATT ACATTA CATTAA ATTA...
108621,1,TGAGTG GAGTGG AGTGGG GTGGGT TGGGTG GGGTGC GGTG...
7541,0,GGAGTA GAGTAG AGTAGA GTAGAA TAGAAT AGAATT GAAT...
...,...,...
96420,1,TGTTTT GTTTTT TTTTTG TTTTGA TTTGAA TTGAAG TGAA...
111689,1,TGAAAT GAAATA AAATAC AATACT ATACTG TACTGA ACTG...
24879,0,CCTGCT CTGCTC TGCTCT GCTCTC CTCTCT TCTCTG CTCT...
92206,1,GTAAAA TAAAAA AAAAAA AAAAAA AAAAAA AAAAAA AAAA...


In [24]:
test_human_enhancer

,label,sequence
20622,1,GGGGAT GGGATC GGATCT GATCTG ATCTGT TCTGTT CTGT...
2851,0,TGCCAG GCCAGT CCAGTT CAGTTC AGTTCC GTTCCT TTCC...
20030,1,CCACTT CACTTC ACTTCC CTTCCA TTCCAA TCCAAC CCAA...
19729,1,TGGCGG GGCGGT GCGGTG CGGTGC GGTGCA GTGCAG TGCA...
27326,1,TCTCTG CTCTGT TCTGTT CTGTTT TGTTTT GTTTTT TTTT...
...,...,...
15162,1,TGAGGA GAGGAG AGGAGC GGAGCT GAGCTT AGCTTG GCTT...
17574,1,GACTTG ACTTGG CTTGGG TTGGGG TGGGGT GGGGTG GGGT...
26804,1,TGAATC GAATCC AATCCA ATCCAG TCCAGC CCAGCA CAGC...
2429,0,TTACTG TACTGG ACTGGC CTGGCT TGGCTC GGCTCC GCTC...


Put in the format DNABERT expects:

In [25]:
test_human_enhancer.index = test_human_enhancer.sequence

In [26]:
test_human_enhancer.drop('sequence', axis=1, inplace=True)

In [27]:
test_human_enhancer

,label
sequence,
GGGGAT GGGATC GGATCT GATCTG ATCTGT TCTGTT CTGTTT TGTTTG GTTTGT TTTGTT TTGTTG TGTTGG GTTGGG TTGGGA TGGGAG GGGAGA GGAGAC GAGACC AGACCC GACCCA ACCCAG CCCAGC CCAGCG CAGCGT AGCGTG GCGTGT CGTGTG GTGTGC TGTGCG GTGCGG TGCGGC GCGGCT CGGCTG GGCTGT GCTGTA CTGTAA TGTAAA GTAAAT TAAATG AAATGT AATGTT ATGTTC TGTTCC GTTCCA TTCCAT TCCATG CCATGG CATGGC ATGGCT TGGCTT GGCTTC GCTTCT CTTCTG TTCTGA TCTGAT CTGATA TGATAC GATACA ATACAT TACATT ACATTT CATTTC ATTTCT TTTCTT TTCTTT TCTTTA CTTTAC TTTACG TTACGA TACGAC ACGACT CGACTC GACTCG ACTCGA CTCGAA TCGAAA CGAAAA GAAAAA AAAAAT AAAATT AAATTT AATTTC ATTTCT TTTCTC TTCTCC TCTCCT CTCCTT TCCTTC CCTTCC CTTCCC TTCCCT TCCCTC CCCTCT CCTCTG CTCTGG TCTGGG CTGGGT TGGGTG GGGTGG GGTGGA GTGGAG TGGAGA GGAGAC GAGACC AGACCA GACCAA ACCAAG CCAAGC CAAGCC AAGCCC AGCCCT GCCCTA CCCTAC CCTACC CTACCC TACCCC ACCCCT CCCCTC CCCTCA CCTCAG CTCAGG TCAGGC CAGGCC AGGCCC GGCCCC GCCCCT CCCCTT CCCTTA CCTTAG CTTAGG TTAGGG TAGGGT AGGGTT GGGTTC GGTTCC GTTCCC TTCCCC TCCCCT CCCCTG CCCTGT CCTGTA CTGTAA TGTAAG GTAAGA TAAGAG AAGAGC AGAGCG GAGCGT AGCGTG GCGTGT CGTGTG GTGTGA TGTGAC GTGACG TGACGC GACGCA ACGCAG CGCAGA GCAGAT CAGATC AGATCC GATCCG ATCCGT TCCGTT CCGTTC CGTTCT GTTCTT TTCTTT TCTTTC CTTTCC TTTCCC TTCCCC TCCCCG CCCCGA CCCGAC CCGACA CGACAA GACAAG ACAAGG CAAGGA AAGGAG AGGAGC GGAGCT GAGCTG AGCTGC GCTGCC CTGCCA TGCCAC GCCACT CCACTG CACTGT ACTGTA CTGTAC TGTACT GTACTG TACTGG ACTGGG CTGGGT TGGGTC GGGTCC GGTCCT GTCCTC TCCTCC CCTCCC CTCCCC TCCCCA CCCCAG CCCAGC CCAGCA CAGCAG AGCAGA GCAGAG CAGAGG AGAGGT GAGGTT AGGTTT GGTTTA GTTTAC TTTACC TTACCA TACCAA ACCAAG CCAAGC CAAGCC AAGCCT AGCCTC GCCTCC CCTCCC CTCCCT TCCCTC CCCTCC CCTCCT CTCCTT TCCTTT CCTTTA CTTTAC TTTACC TTACCC TACCCC ACCCCG CCCCGC CCCGCC CCGCCC CGCCCA GCCCAG CCCAGG CCAGGG CAGGGG AGGGGC GGGGCC GGGCCA GGCCAC GCCACG CCACGA CACGAC ACGACC CGACCG GACCGG ACCGGC CCGGCT CGGCTC GGCTCT GCTCTC CTCTCC TCTCCC CTCCCA TCCCAT CCCATT CCATTC CATTCT ATTCTT TTCTTC TCTTCC CTTCCT TTCCTA TCCTAA CCTAAA CTAAAC TAAACG AAACGA AACGAA ACGAAT CGAATG GAATGG AATGGG ATGGGC TGGGCC GGGCCC GGCCCC GCCCCA CCCCAG CCCAGG CCAGGG CAGGGC AGGGCG GGGCGG GGCGGG GCGGGA CGGGAG GGGAGA GGAGAG GAGAGT AGAGTG GAGTGG AGTGGG GTGGGA TGGGAG GGGAGG GGAGGA GAGGAG AGGAGG GGAGGC GAGGCC AGGCCA GGCCAA GCCAAA CCAAAG CAAAGC AAAGCC AAGCCT AGCCTG GCCTGA CCTGAG CTGAGA TGAGAG GAGAGG AGAGGG GAGGGA AGGGAT GGGATG GGATGA GATGAA ATGAAC TGAACA GAACAC AACACT ACACTA CACTAG ACTAGA CTAGAC TAGACC AGACCC GACCCT ACCCTT CCCTTC CCTTCC CTTCCG TTCCGG TCCGGC CCGGCC CGGCCA GGCCAG GCCAGA CCAGAG CAGAGA AGAGAA GAGAAA AGAAAC GAAACG AAACGG AACGGA ACGGAA CGGAAT GGAATG GAATGC AATGCA ATGCAG TGCAGG GCAGGA CAGGAG AGGAGG GGAGGC GAGGCC AGGCCC GGCCCT GCCCTA CCCTAC CCTACG CTACGC TACGCT ACGCTG CGCTGC GCTGCG CTGCGG TGCGGT GCGGTG CGGTGG GGTGGC GTGGCA TGGCAG GGCAGA GCAGAG CAGAGG AGAGGG GAGGGG AGGGGG GGGGGC GGGGCA GGGCAA GGCAAG GCAAGA CAAGAC AAGACT AGACTG GACTGG ACTGGC CTGGCC TGGCCC GGCCCC GCCCCG CCCCGG CCCGGG CCGGGT CGGGTG GGGTGG GGTGGA GTGGAC TGGACC GGACCA GACCAG ACCAGA CCAGAT CAGATC AGATCA GATCAC ATCACC TCACCA CACCAA ACCAAA CCAAAG CAAAGC AAAGCC AAGCCA AGCCAA GCCAAA CCAAAC CAAACT AAACTC AACTCT ACTCTC CTCTCG TCTCGC CTCGCA TCGCAT CGCATC GCATCT CATCTT ATCTTC TCTTCC CTTCCT TTCCTG TCCTGC CCTGCG CTGCGG TGCGGC GCGGCA CGGCAG GGCAGC GCAGCG CAGCGC AGCGCG GCGCGG CGCGGC GCGGCC CGGCCC GGCCCC GCCCCT CCCCTG CCCTGG CCTGGC CTGGCC TGGCCT GGCCTA GCCTAG CCTAGA CTAGAC TAGACC AGACCT GACCTC ACCTCT CCTCTG CTCTGG TCTGGG CTGGGC TGGGCA GGGCAC GGCACT GCACTC CACTCT ACTCTA CTCTAC TCTACA CTACAG TACAGC ACAGCC CAGCCG AGCCGG,1
TGCCAG GCCAGT CCAGTT CAGTTC AGTTCC GTTCCT TTCCTG TCCTGA CCTGAG CTGAGT TGAGTA GAGTAA AGTAAT GTAATC TAATCT AATCTG ATCTGC TCTGCT CTGCTG TGCTGT GCTGTG CTGTGG TGTGGT GTGGTG TGGTGG GGTGGT GTGGTT TGGTTT GGTTTT GTTTTT TTTTTA TTTTAT TTTATC TTATCA TATCAT ATCATG TCATGG CATGGC ATGGCA TGGCAA GGCAAA GCAAAT CAAATC AAATCA AATCAG ATCAGC TCAGCC CAGCCA AGCCAA GCCAAG CCAAGC CAAGCT AAGCTA AGCTAA GCTAAA CTAAAA TAAAAC AAAACT AAACTA AACTAC ACTACT CTACTT TACTTT ACTTTT CTTTTC TTTTCC TTTCCC TTCCCA TCCCAG CCCAGA CCAGAA CAGAA

In [28]:
train_human_enhancer.index = train_human_enhancer.sequence

In [29]:
train_human_enhancer.drop('sequence', axis=1, inplace=True)

In [30]:
train_human_enhancer

,label
sequence,
GTGCAG TGCAGA GCAGAA CAGAAA AGAAAG GAAAGA AAAGAC AAGACT AGACTG GACTGC ACTGCC CTGCCT TGCCTG GCCTGT CCTGTC CTGTCC TGTCCC GTCCCT TCCCTC CCCTCG CCTCGT CTCGTT TCGTTA CGTTAT GTTATG TTATGG TATGGT ATGGTG TGGTGA GGTGAG GTGAGA TGAGAA GAGAAG AGAAGA GAAGAA AAGAAC AGAACG GAACGG AACGGG ACGGGG CGGGGA GGGGAG GGGAGC GGAGCA GAGCAG AGCAGT GCAGTG CAGTGA AGTGAG GTGAGG TGAGGA GAGGAA AGGAAA GGAAAG GAAAGG AAAGGG AAGGGA AGGGAC GGGACG GGACGT GACGTA ACGTAG CGTAGC GTAGCT TAGCTG AGCTGG GCTGGA CTGGAT TGGATC GGATCA GATCAT ATCATG TCATGG CATGGG ATGGGG TGGGGA GGGGAA GGGAAC GGAACA GAACAA AACAAG ACAAGT CAAGTT AAGTTG AGTTGG GTTGGA TTGGAT TGGATT GGATTC GATTCT ATTCTA TTCTAC TCTACC CTACCG TACCGG ACCGGG CCGGGT CGGGTG GGGTGT GGTGTG GTGTGG TGTGGG GTGGGG TGGGGA GGGGAA GGGAAA GGAAAG GAAAGG AAAGGG AAGGGG AGGGGA GGGGAG GGGAGG GGAGGG GAGGGG AGGGGA GGGGAG GGGAGT GGAGTT GAGTTC AGTTCC GTTCCC TTCCCA TCCCAG CCCAGG CCAGGT CAGGTA AGGTAG GGTAGG GTAGGG TAGGGA AGGGAA GGGAAA GGAAAT GAAATC AAATCC AATCCC ATCCCA TCCCAG CCCAGC CCAGCA CAGCAT AGCATG GCATGA CATGAG ATGAGC TGAGCA GAGCAA AGCAAA GCAAAA CAAAAG AAAAGC AAAGCC AAGCCC AGCCCG GCCCGT CCCGTA CCGTAG CGTAGG GTAGGC TAGGCA AGGCAG GGCAGG GCAGGA CAGGAA AGGAAT GGAATT GAATTT AATTTG ATTTGA TTTGAG TTGAGC TGAGCA GAGCAG AGCAGC GCAGCT CAGCTT AGCTTG GCTTGC CTTGCT TTGCTT TGCTTA GCTTAA CTTAAG TTAAGA TAAGAG AAGAGT AGAGTT GAGTTG AGTTGG GTTGGG TTGGGA TGGGAA GGGAAC GGAACT GAACTC AACTCC ACTCCC CTCCCA TCCCAC CCCACA CCACAA CACAAG ACAAGA CAAGAA AAGAAT AGAATT GAATTG AATTGA ATTGAA TTGAAA TGAAAA GAAAAC AAAACC AAACCT AACCTA ACCTAA CCTAAA CTAAAA TAAAAA AAAAAT AAAATC AAATCC AATCCA ATCCAT TCCATA CCATAG CATAGG ATAGGA TAGGAA AGGAAA GGAAAC GAAACT AAACTG AACTGT ACTGTA CTGTAC TGTACA GTACAC TACACA ACACAA CACAAA ACAAAT CAAATG AAATGG AATGGT ATGGTG TGGTGG GGTGGA GTGGAA TGGAAA GGAAAC GAAACA AAACAC AACACC ACACCC CACCCC ACCCCA CCCCAA CCCAAA CCAAAT CAAATG AAATGT AATGTC ATGTCC TGTCCG GTCCGT TCCGTC CCGTCG CGTCGA GTCGAC TCGACA CGACAG GACAGA ACAGAC CAGACA AGACAA GACAAG ACAAGT CAAGTG AAGTGG AGTGGA GTGGAT TGGATG GGATGA GATGAA ATGAAC TGAACA GAACAA AACAAA ACAAAA CAAAAT AAAATC AAATCC AATCCG ATCCGG TCCGGT CCGGTC CGGTCT GGTCTA GTCTAT TCTATC CTATCC TATCCA ATCCAT TCCATA CCATAC CATACA ATACAA TACAAC ACAACG CAACGG AACGGA ACGGAG,1
TGGGGG GGGGGC GGGGCA GGGCAG GGCAGG GCAGGG CAGGGC AGGGCA GGGCAT GGCATA GCATAG CATAGC ATAGCC TAGCCA AGCCAA GCCAAA CCAAAC CAAACA AAACAA AACAAA ACAAAA CAAAAG AAAAGG AAAGGC AAGGCA AGGCAG GGCAGC GCAGCA CAGCAG AGCAGA GCAGAA CAGAAA AGAAAC GAAACC AAACCT AACCTC ACCTCT CCTCTG CTCTGC TCTGCA CTGCAG TGCAGA GCAGAC CAGACT AGACTT GACTTA ACTTAA CTTAAA TTAAAT TAAATG AAATGT AATGTC ATGTCC TGTCCC GTCCCT TCCCTG CCCTGC CCTGCC CTGCCT TGCCTG GCCTGA CCTGAC CTGACA TGACAG GACAGC ACAGCT CAGCTT AGCTTC GCTTCG CTTCGA TTCGAA TCGAAG CGAAGA GAAGAG AAGAGA AGAGAG GAGAGT AGAGTA GAGTAG AGTAGT GTAGTG TAGTGG AGTGGT GTGGTC TGGTCC GGTCCA GTCCAC TCCACC CCACCC CACCCA ACCCAG CCCAGC CCAGCA CAGCAT AGCATG GCATGG CATGGA ATGGAG TGGAGT GGAGTT GAGTTT AGTTTG GTTTGA TTTGAG TTGAGA TGAGAT GAGATC AGATCT GATCTG ATCTGA TCTGAG CTGAGA TGAGAA GAGAAC AGAACA GAACAG AACAGA ACAGAC CAGACA AGACAG GACAGA ACAGAC CAGACT AGACTG GACTGC ACTGCC CTGCCT TGCCTC GCCTCC CCTCCT CTCCTC TCCTCA CCTCAA CTCAAG TCAAGT CAAGTG AAGTGG AGTGGG GTGGGT TGGGTC GGGTCC GGTCCC GTCCCT TCCCTG CCCTGA CCTGAC CTGACC TGACCC GACCCC ACCCCC CCCCCG CCCCGA CCCGAG CCGAGT CGAGTA GAGTAG AGTAGC GTAGCC TAGCCT AGCCTA GCCTAA CCTAAC CTAACT TAACTG AACTGG ACTGGG CTGGGA TGGGAG GGGAGG GGAGGC GAGGCA AGGCAC GGCACC GCACCC CACCCC ACCCCC CCCCCC CCCCCA CCCCAA CCCAAG CCAAGT CAAGTG AAGTGA AGTGAG GTGAGA TGAGAG GAGAGG AGAGGT GAGGTG AGGTGA GGTGAC GTGACA TGACAG GACAGT ACAGTG CAGTGT AGTGTG GTGTGC TGTGCT GTGCTG TGCTGG GCTGGC CTGGCA TGGCAG GGCAGT GCAGTC CAGTCC AGTCCT GTCCTC TCCTCA CCTCAC CTCACA TCACAG CACAGC ACAGCC CAGCCC AGCCCT GCCCTC CCCTCA CCTCAC CTCACT TCACTC CACTCA ACTCAC CTCACT TCACTC CACTCT ACTCTT CTCTTG TCTTGG CTTGGC TTGGCG TGGCGC GGCGCC GCGCCT CGCCTC GCCTCC CCTCCT CTCCTC TCCTCG CCTCGG CTCGGC TCGGCC CGGCCT GGCCTC GCCTCG CCTCGG CTCGGC TCGGCA CGGCAT GGCATC GCATCC CATCCA ATCCAC TCCAC

In [31]:
# distribution of labels
df = train_human_enhancer
label_distribution = df['label'].value_counts()

In [32]:
(label_distribution)

label
1    57827
0    57768
Name: count, dtype: int64

In [33]:
# distribution of labels
df = test_human_enhancer
label_distribution = df['label'].value_counts()

In [34]:
(label_distribution)

label
1    14475
0    14466
Name: count, dtype: int64

In [ ]:
# saving as tsv file
train_human_enhancer.to_csv('train.tsv', sep="\t")

In [ ]:
# saving as tsv file
test_human_enhancer.to_csv('dev.tsv', sep="\t")